# Merge AutoML Submissions
Finds all individual task CSVs in the current folder and merges them into a single `final_submission.csv`.

In [ ]:
import os
import glob
import re
import pandas as pd
from functools import reduce

In [ ]:
# Find all task CSVs in the current (submission) directory
csv_files = glob.glob('task_*.csv')
print(f"Found {len(csv_files)} files to merge: {csv_files}\n")

if csv_files:
    # Read each CSV into a list of DataFrames
    dfs = [pd.read_csv(f) for f in csv_files]
    
    # Merge all DataFrames dynamically on 'Participant_ID' using an outer join
    final_df = reduce(lambda left, right: pd.merge(left, right, on='Participant_ID', how='outer'), dfs)
    
    # Task 4.4 = geometric mean of Task 4.1, 4.2, 4.3 (no separate CSV)
    final_df['Task_4.4'] = (final_df['Task_4.1'] * final_df['Task_4.2'] * final_df['Task_4.3']) ** (1/3)

    # Sort columns: Participant_ID first, then numerically by task number
    def sort_cols(col_name):
        if col_name == 'Participant_ID':
            return -1
        match = re.search(r'4[._](\d+)', col_name)
        return int(match.group(1)) if match else 999
        
    final_df = final_df[sorted(final_df.columns, key=sort_cols)]
    
    # Export to final_submission.csv
    final_df.to_csv('final_submission.csv', index=False)
    print("Successfully saved merged data to 'final_submission.csv'!")
    display(final_df)
else:
    print("No 'task_*.csv' files found in the current directory.")